# 01 — Load & Inspect LIAR

**Goal:** load LIAR train/valid/test TSV files and inspect:
- dataset shapes
- label distribution
- sample `statement` texts
- sample `context` texts
- simple text length statistics

**Expected data path (edit if needed):**
- `data/liar/train.tsv`
- `data/liar/valid.tsv`
- `data/liar/test.tsv`

This notebook is the user's own working inspection notebook for this project.

## 0) Setup

Edit `DATA_DIR` only if your LIAR TSV files are stored elsewhere.

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime

DATA_DIR = Path("data/liar")  # change if needed
train_path = DATA_DIR / "train.tsv"
valid_path = DATA_DIR / "valid.tsv"
test_path  = DATA_DIR / "test.tsv"

cols = [
    "id", "label", "statement", "subject", "speaker", "speaker_job", "state", "party",
    "barely_true_counts", "false_counts", "half_true_counts", "mostly_true_counts",
    "pants_on_fire_counts", "context"
]

def load_tsv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"File not found: {path}.\n"
            f"Please upload LIAR TSV files to {DATA_DIR}/ or change DATA_DIR."
        )
    return pd.read_csv(path, sep="\t", header=None, names=cols)

train = load_tsv(train_path)
valid = load_tsv(valid_path)
test  = load_tsv(test_path)

print("Loaded LIAR TSV files successfully.")

## 1) Basic shapes

Print the number of rows/columns for each split.

In [ ]:
print("Shapes:")
print("  train:", train.shape)
print("  valid:", valid.shape)
print("  test :", test.shape)

## 2) Label distribution

Show label counts for each split.

In [ ]:
def show_label_counts(df: pd.DataFrame, name: str):
    print(f"\n{name} label counts:")
    print(df["label"].value_counts())

show_label_counts(train, "Train")
show_label_counts(valid, "Valid")
show_label_counts(test, "Test")

## 3) Example statements

Print a few `statement` examples to understand what the main text field looks like.

In [ ]:
print("Sample statements (train):")
for i, s in enumerate(train["statement"].head(5).fillna("").astype(str).tolist(), 1):
    print(f"{i}. {s}")

## 4) Example contexts

The supervisor specifically asked what the `context` field looks like, so inspect it separately.

In [ ]:
print("Sample contexts (train):")
for i, c in enumerate(train["context"].head(5).fillna("").astype(str).tolist(), 1):
    print(f"{i}. {c}")

## 5) Basic text quality checks

Check:
- empty values
- character lengths
- word lengths

In [ ]:
def text_stats(df: pd.DataFrame, col: str, name: str):
    s = df[col].fillna("").astype(str)
    empty = (s.str.strip() == "").sum()
    char_lengths = s.str.len()
    word_lengths = s.str.split().str.len()

    print(f"\n{name} - {col} stats:")
    print("  empty:", int(empty))
    print(
        "  chars (min / median / mean / max):",
        int(char_lengths.min()),
        int(char_lengths.median()),
        float(char_lengths.mean()),
        int(char_lengths.max())
    )
    print(
        "  words (min / median / mean / max):",
        int(word_lengths.min()),
        int(word_lengths.median()),
        float(word_lengths.mean()),
        int(word_lengths.max())
    )

text_stats(train, "statement", "Train")
text_stats(valid, "statement", "Valid")
text_stats(test,  "statement", "Test")

text_stats(train, "context", "Train")

## 6) Optional: inspect one full row

Useful when you want to quickly show the supervisor one full LIAR example.

In [ ]:
sample_idx = 0

print("Sample full row:")
print("id:", train.iloc[sample_idx]["id"])
print("label:", train.iloc[sample_idx]["label"])
print("statement:", train.iloc[sample_idx]["statement"])
print("context:", train.iloc[sample_idx]["context"])

## 7) Optional: save a short overview file

This creates a small markdown summary under `results/` in your runtime.

In [ ]:
def label_counts_str(df: pd.DataFrame) -> str:
    vc = df["label"].value_counts()
    return "\n".join([f"- {k}: {int(v)}" for k, v in vc.items()])

overview = []
overview.append("# LIAR Dataset Overview")
overview.append("")
overview.append(f"- Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
overview.append(f"- Data dir: {DATA_DIR}")
overview.append("")
overview.append("## Shapes")
overview.append(f"- Train: {train.shape}")
overview.append(f"- Valid: {valid.shape}")
overview.append(f"- Test: {test.shape}")
overview.append("")
overview.append("## Label counts (Train)")
overview.append(label_counts_str(train))
overview.append("")
overview.append("## Label counts (Valid)")
overview.append(label_counts_str(valid))
overview.append("")
overview.append("## Label counts (Test)")
overview.append(label_counts_str(test))
overview.append("")
overview.append("## Sample statements (Train)")
for i, s in enumerate(train["statement"].head(3).fillna("").astype(str).tolist(), 1):
    overview.append(f"- {i}. {s}")
overview.append("")
overview.append("## Sample contexts (Train)")
for i, c in enumerate(train["context"].head(3).fillna("").astype(str).tolist(), 1):
    overview.append(f"- {i}. {c}")

out_dir = Path("results")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "liar_overview.md"
out_path.write_text("\n".join(overview), encoding="utf-8")

print("Saved:", out_path.resolve())